# 14. Búsqueda semántica con embeddings preentrenados

En el notebook `13_Ejemplo_embeddings_preentrenados.ipynb` vimos que los embeddings preentrenados representan palabras (o frases) como vectores densos que capturan significado: palabras relacionadas obtienen vectores próximos entre sí.

En este notebook aplicamos esa propiedad a un caso de uso muy habitual en NLP: la **búsqueda semántica**. Dada una colección de documentos y una consulta en lenguaje natural, queremos encontrar los documentos más relevantes para esa consulta, aunque la consulta no comparta ninguna palabra literal con ellos.

Usamos como colección de documentos un corpus de frases deportivas sobre fútbol, baloncesto y tenis, similar al que empleamos en los notebooks `10_LSI_ejemplo_basico.ipynb` y `11_LDA_ejemplo_basico.ipynb`, pero aquí con frases completas en lenguaje natural en lugar de listas de palabras repetidas.

---

## 1. Normalización para embeddings preentrenados

Antes de vectorizar los documentos hay que normalizarlos, pero **de forma muy distinta a como lo hacíamos en las Unidades 1 y 2**. Allí, para preparar el texto de cara a Bag of Words, TF-IDF, LSI o LDA, aplicábamos con spaCy una normalización agresiva: pasar a minúsculas, eliminar *stopwords*, lematizar cada palabra y quitar acentos y signos de puntuación. Esa normalización tiene sentido cuando el objetivo es reducir el vocabulario y quedarnos solo con las palabras más informativas.

Con un modelo de embeddings preentrenado el objetivo es el contrario: el modelo fue entrenado sobre texto natural, con sus artículos, sus preposiciones, sus verbos conjugados y sus tildes, así que cuanto más se parezca nuestro texto a ese tipo de texto natural, mejor funcionará. Lematizar o eliminar *stopwords* aleja el texto de la distribución con la que se entrenó el modelo, en lugar de ayudarle.

Antes de decidir qué normalización aplicar, comprobamos empíricamente cómo reacciona el modelo a tres transformaciones del texto: cambios de mayúsculas/minúsculas, presencia o ausencia de tildes, y espacios en blanco adicionales.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # silencia los mensajes informativos de TensorFlow

import tensorflow as tf
tf.get_logger().setLevel('ERROR')          # silencia los WARNING de la API antigua de TF

import numpy as np
import pandas as pd
import tensorflow_hub as hub

modelo = hub.load('https://tfhub.dev/google/nnlm-es-dim50/2')   # 50 dimensiones

def similitud_coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [2]:
# Comprobamos si el modelo distingue mayusculas, tildes y espacios extra
pares_prueba = [
    ('mayusculas',    'Messi anoto un gol',                'messi anoto un gol'),
    ('acentos',       'Nadal gano el partido',             'Nadal ganó el partido'),
    ('espacios extra','Nadal ganó el partido de tenis',    'Nadal   ganó   el partido  de tenis'),
]

for etiqueta, texto_a, texto_b in pares_prueba:
    emb_a = modelo([texto_a]).numpy()[0]
    emb_b = modelo([texto_b]).numpy()[0]
    print(f'{etiqueta:15s} -> similitud = {similitud_coseno(emb_a, emb_b):.4f}   ({texto_a!r} vs {texto_b!r})')

mayusculas      -> similitud = 0.9284   ('Messi anoto un gol' vs 'messi anoto un gol')
acentos         -> similitud = 0.9418   ('Nadal gano el partido' vs 'Nadal ganó el partido')
espacios extra  -> similitud = 1.0000   ('Nadal ganó el partido de tenis' vs 'Nadal   ganó   el partido  de tenis')


Los resultados muestran que el modelo **sí distingue entre mayúsculas y minúsculas**, y **sí distingue entre palabras con y sin tilde**: en ambos casos la similitud es alta pero no llega a 1.0 (en torno a 0.93), así que "Messi" y "messi", o "gano" y "ganó", no son exactamente lo mismo para el modelo, aunque tampoco los trata como palabras completamente distintas. En cambio, los **espacios en blanco adicionales no afectan al resultado** (similitud igual a 1.0000): el tokenizador interno del modelo ya los normaliza por su cuenta.

De aquí se deduce la normalización adecuada para este caso: pasamos el texto a minúsculas para no depender de cómo esté escrito, mantenemos los acentos exactamente como se escriben en español, y no hace falta preocuparnos por espacios adicionales. No aplicamos lematización ni eliminamos *stopwords*, a diferencia de las Unidades 1 y 2.

In [3]:
def normalizar(texto):
    """Normalizacion ligera para embeddings: solo minusculas y espacios limpios."""
    return ' '.join(texto.strip().lower().split())

---

## 2. Colección de documentos

Definimos un corpus de once frases deportivas: cuatro sobre fútbol, cuatro sobre baloncesto y tres sobre tenis, con nombres de deportistas reales para hacerlas más reconocibles.

In [4]:
documentos = [
    # Futbol
    'Messi recibió el balón en el área y remató de zurda para anotar el gol que decidió el partido.',
    'El árbitro señaló un penalti muy polémico después de una entrada dura de Ronaldo dentro del área rival.',
    'El centrocampista repartió un pase preciso que dejó a Neymar solo ante el portero para marcar a placer.',
    'El equipo capitaneado por Messi levantó la copa de la liga tras ganar la final en su propio estadio.',
    # Baloncesto
    'Lebron anotó una canasta de tres puntos justo antes de que sonara la bocina del último cuarto del partido.',
    'Jordan capturó un rebote decisivo bajo el aro y lo convirtió en un mate espectacular ante su rival.',
    'El entrenador pidió un tiempo muerto para que Lebron se recuperara antes del tiro libre decisivo del encuentro.',
    'El alero robó el balón en el perímetro y corrió a machacar en un contraataque muy rápido junto a Jordan.',
    # Tenis
    'Nadal atacó con un golpe potente desde el fondo de la pista y cerró el punto sin que su rival pudiera reaccionar.',
    'Federer remontó dos sets en contra y se llevó el partido en el quinto set de la final.',
    'La final femenina de Wimbledon se decidió con un intercambio de golpes muy intenso que terminó en un remate ganador cerca de la malla.',
]

print(f'Documentos: {len(documentos)} (futbol: 4, baloncesto: 4, tenis: 3)')

Documentos: 11 (futbol: 4, baloncesto: 4, tenis: 3)


---

## 3. Consulta del usuario

La nueva frase o consulta habla claramente de tenis, pero usa un jugador y un vocabulario técnico —"revés", "paralelo", "red", "volea", "Roland Garros"— que **no aparecen en ninguno de los once documentos**. Con Bag of Words o TF-IDF esta consulta no compartiría vocabulario con los documentos de tenis y sería difícil relacionarla con ellos. El objetivo es comprobar si, con embeddings, el modelo es capaz de relacionarla semánticamente con los documentos de tenis a pesar de no compartir ninguna palabra.

In [5]:
consulta = 'Djokovic golpeó un potente revés paralelo y se acercó a la red para cerrar el set con una volea ganadora en Roland Garros.'

print(f'Consulta: {consulta!r}')

Consulta: 'Djokovic golpeó un potente revés paralelo y se acercó a la red para cerrar el set con una volea ganadora en Roland Garros.'


---

## 4. Búsqueda semántica

Normalizamos documentos y consulta, obtenemos su embedding con el modelo, y calculamos la similitud coseno de la consulta contra cada documento. Ordenamos los documentos de mayor a menor similitud: los primeros de la lista son los que el modelo considera más relevantes para la consulta.

In [6]:
documentos_norm = [normalizar(doc) for doc in documentos]
consulta_norm   = normalizar(consulta)

emb_docs     = modelo(documentos_norm).numpy()
emb_consulta = modelo([consulta_norm]).numpy()[0]

resultados = sorted(
    [(doc, similitud_coseno(emb_consulta, emb_docs[i])) for i, doc in enumerate(documentos)],
    key=lambda x: x[1], reverse=True
)

print(f"Consulta: '{consulta}'\n")
print('Documentos ordenados por relevancia semantica:')
for doc, sim in resultados:
    print(f'  {sim:.3f}  ->  {doc}')

Consulta: 'Djokovic golpeó un potente revés paralelo y se acercó a la red para cerrar el set con una volea ganadora en Roland Garros.'

Documentos ordenados por relevancia semantica:
  0.787  ->  Federer remontó dos sets en contra y se llevó el partido en el quinto set de la final.
  0.780  ->  Nadal atacó con un golpe potente desde el fondo de la pista y cerró el punto sin que su rival pudiera reaccionar.
  0.739  ->  La final femenina de Wimbledon se decidió con un intercambio de golpes muy intenso que terminó en un remate ganador cerca de la malla.
  0.725  ->  Jordan capturó un rebote decisivo bajo el aro y lo convirtió en un mate espectacular ante su rival.
  0.718  ->  El alero robó el balón en el perímetro y corrió a machacar en un contraataque muy rápido junto a Jordan.
  0.684  ->  El equipo capitaneado por Messi levantó la copa de la liga tras ganar la final en su propio estadio.
  0.651  ->  Messi recibió el balón en el área y remató de zurda para anotar el gol que decidió e

## Conclusiones sobre el resultado con 50 dimensiones

Los tres primeros documentos son, en este orden, la frase de Federer, la de Nadal y la de la final de Wimbledon: las tres frases de tenis quedan por delante de cualquier frase de fútbol o de baloncesto, y ninguna de ellas comparte una sola palabra con la consulta sobre Djokovic. Esto confirma la idea central del notebook: el modelo relaciona los textos por significado, no por coincidencia literal de términos.

Nótese que la frase mejor situada no es literalmente la más parecida en vocabulario ni la más "obvia": es la de Federer, seguida muy de cerca por la de Nadal (0.787 frente a 0.780), y algo más lejos la de Wimbledon (0.739). Justo después aparecen ya las primeras frases de baloncesto, con una similitud notablemente menor. El modelo, por tanto, agrupa correctamente las tres frases de tenis en la zona alta del ranking, aunque el orden exacto entre ellas dependa de matices que no controlamos directamente al escribir las frases.

---

## 5. Repetimos la búsqueda con el modelo de 128 dimensiones

Como vimos en el notebook 13, el mismo modelo NNLM está disponible también en una versión de 128 dimensiones, con mayor capacidad para representar matices de significado. Repetimos exactamente el mismo proceso, cambiando únicamente la URL del modelo.

In [7]:
modelo_128 = hub.load('https://tfhub.dev/google/nnlm-es-dim128/2')   # 128 dimensiones

emb_docs_128     = modelo_128(documentos_norm).numpy()
emb_consulta_128 = modelo_128([consulta_norm]).numpy()[0]

resultados_128 = sorted(
    [(doc, similitud_coseno(emb_consulta_128, emb_docs_128[i])) for i, doc in enumerate(documentos)],
    key=lambda x: x[1], reverse=True
)

print(f"Consulta: '{consulta}'\n")
print('Documentos ordenados por relevancia semantica (128 dimensiones):')
for doc, sim in resultados_128:
    print(f'  {sim:.3f}  ->  {doc}')

Consulta: 'Djokovic golpeó un potente revés paralelo y se acercó a la red para cerrar el set con una volea ganadora en Roland Garros.'

Documentos ordenados por relevancia semantica (128 dimensiones):
  0.772  ->  Nadal atacó con un golpe potente desde el fondo de la pista y cerró el punto sin que su rival pudiera reaccionar.
  0.758  ->  La final femenina de Wimbledon se decidió con un intercambio de golpes muy intenso que terminó en un remate ganador cerca de la malla.
  0.689  ->  Federer remontó dos sets en contra y se llevó el partido en el quinto set de la final.
  0.639  ->  Jordan capturó un rebote decisivo bajo el aro y lo convirtió en un mate espectacular ante su rival.
  0.623  ->  El alero robó el balón en el perímetro y corrió a machacar en un contraataque muy rápido junto a Jordan.
  0.612  ->  El árbitro señaló un penalti muy polémico después de una entrada dura de Ronaldo dentro del área rival.
  0.588  ->  Messi recibió el balón en el área y remató de zurda para anotar

## Conclusiones finales

Con 128 dimensiones, las tres frases de tenis vuelven a ocupar las tres primeras posiciones, otra vez muy por delante de fútbol y baloncesto, pero el orden interno cambia: ahora Nadal queda primero, seguido de la final de Wimbledon y, en tercer lugar, Federer. Es decir, ambos modelos identifican correctamente el mismo grupo de documentos relevantes, pero no coinciden exactamente en cuál de ellos es "el más relevante" dentro de ese grupo. Esto es un buen recordatorio de que dos modelos de embeddings distintos pueden estar de acuerdo en lo importante —el tema general de la consulta— y, aun así, discrepar en matices finos de orden, porque cada uno ha aprendido una representación ligeramente distinta a partir de datos y arquitecturas distintas.

De este notebook nos llevamos dos ideas importantes:

1. **La normalización para embeddings preentrenados debe ser ligera**: minúsculas y poco más, muy distinta de la normalización agresiva de las Unidades 1 y 2 (lematización, eliminación de *stopwords*, eliminación de acentos), porque el modelo funciona mejor cuanto más se parezca el texto de entrada al lenguaje natural con el que fue entrenado.

2. **La búsqueda semántica agrupa por significado, no por vocabulario compartido**: en ambos modelos, las tres frases de tenis quedan claramente por delante de las de fútbol y baloncesto para una consulta que no comparte ni una palabra con ellas. El número de dimensiones del embedding (50 o 128) puede cambiar el orden fino entre documentos igualmente relevantes, pero en este caso no cambia la conclusión principal: el modelo relaciona correctamente la consulta con el grupo temático adecuado.

---

## 6. Un embedding por frase, no uno por palabra

Puede resultar un poco contraintuitivo: si una frase tiene 18 palabras, ¿no debería el modelo devolver 18 vectores, uno por palabra, en lugar de un único vector de 50 posiciones? Vamos a comprobarlo con la frase de `Federer remontó dos sets en contra y se llevó el partido en el quinto set de la final`, comparando la forma del resultado cuando le pasamos la frase completa frente a cuando le pasamos sus palabras por separado.

In [8]:
frase_federer = documentos_norm[9]   # 'federer remonto dos sets en contra ...'
palabras = frase_federer.split()

emb_frase    = modelo([frase_federer]).numpy()   # pasamos la frase completa
emb_palabras = modelo(palabras).numpy()          # pasamos las palabras sueltas, una por una

print(f'Frase: {frase_federer!r}')
print(f'Numero de palabras de la frase: {len(palabras)}\n')

print(f'Forma al pasar la frase completa:   {emb_frase.shape}    -> 1 vector de 50 posiciones')
print(f'Forma al pasar las palabras sueltas: {emb_palabras.shape}  -> {len(palabras)} vectores de 50 posiciones, uno por palabra\n')

# Comparamos el embedding de la frase con la media de los embeddings de sus palabras
media_palabras = emb_palabras.mean(axis=0)
print(f'Similitud coseno entre el embedding de la frase y la media de sus {len(palabras)} palabras: '
      f'{similitud_coseno(emb_frase[0], media_palabras):.4f}')

Frase: 'federer remontó dos sets en contra y se llevó el partido en el quinto set de la final.'
Numero de palabras de la frase: 18

Forma al pasar la frase completa:   (1, 50)    -> 1 vector de 50 posiciones
Forma al pasar las palabras sueltas: (18, 50)  -> 18 vectores de 50 posiciones, uno por palabra

Similitud coseno entre el embedding de la frase y la media de sus 18 palabras: 1.0000


* Si nos fijamos, la similitud entre el embedding de la frase y el embedding formado por la media de las palabras tiene una similutud de 1.0; es decir una similitud máxima, pero los embeddings resultantes no son exactamente iguales:

```text
media_palabras = [ 0.07832454, -0.09244379,  0.01333476, -0.05164124,  0.00776393, ...]

emb_frase =      [ 0.33230293, -0.39220583,  0.0565746 , -0.21909525,  0.03293955, ...]
```

Esto se debe a que emb_frase se calcula como la suma de los embeddings de las `n` palabras de la frase dividida entre la **raíz cuadrada de `n`**, y no como la media aritmética simple (dividida entre `n`):

$$ \text{embedding}_{\text{frase}} = \frac{1}{\sqrt{n}} \sum_{i=1}^{n} \text{embedding}_{\text{palabra}_i} $$

Esta fórmula se conoce como combinador **`sqrtn`**, y es una técnica habitual en los modelos de tipo *bag-of-embeddings* (anteriores a los transformers) para combinar los embeddings de varias palabras en un único vector de longitud fija.

¿Por qué dividir entre `√n` y no entre `n`? Es un punto intermedio entre dos extremos poco deseables:

* **No dividir (suma pura)**: el vector de una frase de 18 palabras tendría una magnitud mucho mayor que el de una frase de 5 palabras, solo por tener más términos que sumar, sin que eso refleje ninguna diferencia real de significado.
* **Dividir entre `n` (media aritmética)**: el peso de cada palabra se diluye demasiado deprisa a medida que crece la frase. En una frase de 20 palabras, una palabra clave pesaría solo `1/18` del vector final; en una frase de 5 palabras, esa misma palabra pesaría `1/5`, cuatro veces más.

Dividir entre `√n` crece más despacio que dividir entre `n` (en nuestra frase de Federer, `√18 ≈ 4.24` frente a `n = 18`), así que amortigua ambos problemas a la vez: evita que la magnitud se dispare con las frases largas, sin diluir tanto el significado como lo haría una media pura.

Esta es la razón por la que `emb_frase` y `media_palabras` **apuntan en la misma dirección** —contienen la misma información semántica, de ahí la similitud coseno de 1.0— pero **no tienen los mismos valores**: `emb_frase` es `media_palabras` multiplicado por `√n` (por `√18 ≈ 4.24` en nuestro ejemplo).

## Conclusión

El modelo siempre devuelve **un vector de 50 posiciones por cada texto que le pasamos**, sea ese texto una sola palabra o una frase de dieciocho, y lo construye combinando los embeddings de las palabras de la frase con la fórmula `sqrtn` que hemos visto arriba.

Una última precisión importante: como la búsqueda semántica de este notebook se basa en la **similitud coseno**, que solo depende de la dirección de un vector y no de su magnitud, este factor de escala (`√n`) no afecta a ninguno de los resultados de la búsqueda semántica que hemos visto en las secciones anteriores. Sí sería relevante si comparásemos los documentos con una distancia euclídea en lugar de con similitud coseno.